# Session 3, Module 05: Decorators


This module covers:
- Functions as first-class objects
- Closures
- Basic decorator pattern
- Decorators with arguments
- functools.wraps — preserving metadata
- Practical decorators: @timer, @retry, @log_execution, @validate_input

Data Engineering Context:
Decorators add cross-cutting concerns like retry logic, execution timing,
and logging without modifying the core function code.


In [1]:
import functools
import time
import random
from typing import Callable, Any

## Functions As First-Class Objects


In [3]:
print("=== Functions as First-Class Objects ===")


def greet(name: str) -> str:
    """Simple greeting function."""
    return f"Hello, {name}!"


# Functions can be assigned to variables
say_hello = greet
print(say_hello("Alice"))

# Functions can be passed as arguments
def apply_twice(func: Callable, value: str) -> str:
    """Apply a function twice."""
    return func(func(value))


def add_exclaim(text: str) -> str:
    return text + "!"


result = apply_twice(add_exclaim, "Hello")
print(f"apply_twice(add_exclaim, 'Hello'): {result}")

# Functions can be returned from other functions
def create_multiplier(factor: int) -> Callable:
    """Create a function that multiplies by factor."""
    def multiplier(x: int) -> int:
        return x * factor
    return multiplier


double = create_multiplier(2)
triple = create_multiplier(3)

print(f"double(5): {double(5)}")
print(f"triple(5): {triple(5)}")

=== Functions as First-Class Objects ===
Hello, Alice!
apply_twice(add_exclaim, 'Hello'): Hello!!
double(5): 10
triple(5): 15


## Closures


In [4]:
print("\n=== Closures ===")


def create_counter(start: int = 0) -> Callable:
    """
    Create a counter function using closure.

    The inner function 'remembers' the count variable from
    the enclosing scope even after create_counter returns.
    """
    count = start

    def counter() -> int:
        nonlocal count
        count += 1
        return count

    return counter


# Create independent counters
counter1 = create_counter(0)
counter2 = create_counter(100)

print(f"counter1(): {counter1()}")  # 1
print(f"counter1(): {counter1()}")  # 2
print(f"counter2(): {counter2()}")  # 101
print(f"counter1(): {counter1()}")  # 3


=== Closures ===
counter1(): 1
counter1(): 2
counter2(): 101
counter1(): 3


## Basic Decorator Pattern


In [ ]:
print("\n=== Basic Decorator Pattern ===")


def simple_decorator(func: Callable) -> Callable:
    """
    A simple decorator that prints before and after function execution.
    """
    def wrapper(*args, **kwargs):
        print(f"Before calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"After calling {func.__name__}")
        return result
    return wrapper


# Apply decorator manually
def say_hello(name: str) -> str:
    return f"Hello, {name}!"


decorated_hello = simple_decorator(say_hello)
print(decorated_hello("World"))

# Apply decorator with @ syntax (same result)
@simple_decorator
def say_goodbye(name: str) -> str:
    return f"Goodbye, {name}!"


print(say_goodbye("World"))

## Preserving Metadata With Functools.Wraps


In [ ]:
print("\n=== Preserving Metadata ===")

# Without @wraps, the wrapper replaces function metadata
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@bad_decorator
def original_function():
    """Original docstring."""
    pass


print(f"Without @wraps:")
print(f"  __name__: {original_function.__name__}")  # 'wrapper'
print(f"  __doc__: {original_function.__doc__}")    # None


# With @wraps, metadata is preserved
def good_decorator(func):
    @functools.wraps(func)  # Preserves func's metadata
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@good_decorator
def decorated_function():
    """Decorated docstring."""
    pass


print(f"\nWith @wraps:")
print(f"  __name__: {decorated_function.__name__}")  # 'decorated_function'
print(f"  __doc__: {decorated_function.__doc__}")    # 'Decorated docstring.'

## Practical Decorator: @Timer


In [ ]:
print("\n=== @timer Decorator ===")


def timer(func: Callable) -> Callable:
    """
    Decorator that measures and prints execution time.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱ {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper


@timer
def slow_operation(n: int) -> int:
    """Simulate a slow operation."""
    time.sleep(0.1)  # Simulate work
    return sum(range(n))


result = slow_operation(1000)
print(f"Result: {result}")

## Practical Decorator: @Retry


In [ ]:
print("\n=== @retry Decorator ===")


def retry(max_attempts: int = 3, delay: float = 1.0):
    """
    Decorator factory that retries a function on failure.

    Args:
        max_attempts: Maximum number of attempts
        delay: Delay between attempts in seconds
    """
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None

            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    print(f"  Attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt < max_attempts:
                        time.sleep(delay)

            raise last_exception

        return wrapper
    return decorator


@retry(max_attempts=3, delay=0.1)
def unreliable_api_call() -> str:
    """Simulate an unreliable API that sometimes fails."""
    if random.random() < 0.7:  # 70% failure rate
        raise ConnectionError("API connection failed")
    return "Success!"


print("Calling unreliable API:")
try:
    result = unreliable_api_call()
    print(f"Result: {result}")
except ConnectionError as e:
    print(f"Final failure: {e}")

## Practical Decorator: @Log Execution


In [ ]:
print("\n=== @log_execution Decorator ===")


def log_execution(func: Callable) -> Callable:
    """
    Decorator that logs function entry, exit, and any exceptions.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        args_repr = [repr(a) for a in args]
        kwargs_repr = [f"{k}={v!r}" for k, v in kwargs.items()]
        signature = ", ".join(args_repr + kwargs_repr)

        print(f"📥 {func.__name__}({signature})")

        try:
            result = func(*args, **kwargs)
            print(f"📤 {func.__name__} returned {result!r}")
            return result
        except Exception as e:
            print(f"❌ {func.__name__} raised {type(e).__name__}: {e}")
            raise

    return wrapper


@log_execution
def process_record(record: dict, validate: bool = True) -> dict:
    """Process a data record."""
    if validate and "id" not in record:
        raise ValueError("Missing 'id' field")
    return {**record, "processed": True}


# Test the decorator
process_record({"id": 1, "name": "Alice"})
process_record({"id": 2}, validate=False)

try:
    process_record({"name": "Bob"})  # Will raise
except ValueError:
    pass

## Practical Decorator: @Validate Input


In [ ]:
print("\n=== @validate_input Decorator ===")


def validate_input(**validators):
    """
    Decorator factory that validates function arguments.

    Args:
        **validators: Mapping of argument names to validator functions
    """
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Get function signature to map args to names
            import inspect
            sig = inspect.signature(func)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()

            # Run validators
            for param_name, validator in validators.items():
                if param_name in bound.arguments:
                    value = bound.arguments[param_name]
                    if not validator(value):
                        raise ValueError(
                            f"Validation failed for '{param_name}': {value!r}"
                        )

            return func(*args, **kwargs)
        return wrapper
    return decorator


# Define validators
def is_positive(x):
    return x > 0


def is_non_empty_string(s):
    return isinstance(s, str) and len(s.strip()) > 0


@validate_input(name=is_non_empty_string, batch_size=is_positive)
def create_pipeline(name: str, batch_size: int = 100) -> dict:
    """Create a pipeline configuration."""
    return {"name": name, "batch_size": batch_size}


# Test
print(create_pipeline("my_pipeline", 500))

try:
    create_pipeline("", 100)  # Empty name
except ValueError as e:
    print(f"Validation error: {e}")

try:
    create_pipeline("pipeline", -5)  # Negative batch size
except ValueError as e:
    print(f"Validation error: {e}")

## Stacking Multiple Decorators


In [ ]:
print("\n=== Stacking Decorators ===")


# Decorators are applied bottom-up (closest to function first)
@timer
@log_execution
@retry(max_attempts=2, delay=0.1)
def complex_operation(value: int) -> int:
    """A complex operation with multiple decorators."""
    if random.random() < 0.5:
        raise ValueError("Random failure")
    return value * 2

This is equivalent to:
complex_operation = timer(log_execution(retry(max_attempts=2)(complex_operation)))

In [ ]:
print("Running complex_operation:")
try:
    result = complex_operation(10)
    print(f"Final result: {result}")
except ValueError as e:
    print(f"Final failure: {e}")

## Class-Based Decorator


In [ ]:
print("\n=== Class-Based Decorator ===")


class CountCalls:
    """
    A class-based decorator that counts function calls.
    """

    def __init__(self, func: Callable):
        functools.update_wrapper(self, func)
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call #{self.count} to {self.func.__name__}")
        return self.func(*args, **kwargs)


@CountCalls
def say_hi(name: str) -> str:
    return f"Hi, {name}!"


say_hi("Alice")
say_hi("Bob")
say_hi("Charlie")
print(f"Total calls: {say_hi.count}")

## Summary


In [ ]:
print("\n=== Summary ===")
print("""
Basic Decorator:
  def decorator(func):
      @functools.wraps(func)
      def wrapper(*args, **kwargs):
          # Before
          result = func(*args, **kwargs)
          # After
          return result
      return wrapper

Decorator with Arguments:
  def decorator_factory(arg):
      def decorator(func):
          @functools.wraps(func)
          def wrapper(*args, **kwargs):
              # Use arg here
              return func(*args, **kwargs)
          return wrapper
      return decorator

  @decorator_factory(arg)
  def function(): ...

Common Patterns:
  @timer          - Measure execution time
  @retry          - Retry on failure
  @log_execution  - Log entry/exit
  @validate_input - Validate arguments
  @cache          - Memoization (use functools.lru_cache)

Best Practices:
  - Always use @functools.wraps
  - Accept *args, **kwargs in wrapper
  - Return the function result
  - Handle exceptions properly
""")